# 04 - Backtest Analysis

Walk-forward backtest of the flagship **multi-asset rotation** strategy, a full **tearsheet**, the **Deflated Sharpe Ratio**, and **BHY multiple-testing** correction across strategy variants.

In [1]:
import sys, os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Put the quantcortex repo root on the path (notebooks live in research/).
for p in ("..", "."):
    ap = os.path.abspath(p)
    if ap not in sys.path:
        sys.path.insert(0, ap)

RNG = np.random.default_rng(42)

def load_prices(symbols, start="2018-01-01", periods=1800):
    """Real prices via yfinance if available, else deterministic synthetic GBM."""
    try:
        from data.providers.yfinance_provider import YFinanceProvider
        px = YFinanceProvider().get_prices(list(symbols), start=start)
        if px is not None and not px.empty and px.shape[0] > 120:
            return px.dropna(how="all").ffill().dropna()
        raise RuntimeError("empty frame")
    except Exception as exc:
        print(f"[offline] yfinance unavailable ({type(exc).__name__}); using synthetic data.")
    dates = pd.bdate_range(start, periods=periods)
    mu = RNG.normal(0.0003, 0.00015, len(symbols))
    vol = RNG.uniform(0.008, 0.018, len(symbols))
    shocks = RNG.normal(mu, vol, size=(periods, len(symbols)))
    return pd.DataFrame(100 * np.exp(np.cumsum(shocks, axis=0)),
                        index=dates, columns=list(symbols))

def synth_ohlcv(close):
    """Build an OHLCV frame around a close series (synthetic intrabar range)."""
    close = close.dropna()
    hi = close * (1 + np.abs(RNG.normal(0, 0.004, len(close))))
    lo = close * (1 - np.abs(RNG.normal(0, 0.004, len(close))))
    op = close.shift(1).fillna(close.iloc[0])
    vol = RNG.integers(1_000_000, 6_000_000, len(close)).astype(float)
    return pd.DataFrame({"open": op, "high": hi, "low": lo, "close": close, "volume": vol})

print("quantcortex research environment ready.")


quantcortex research environment ready.


In [2]:
symbols = ["QQQ","VGT","GLD","TLT","SPY","VIG"]
prices = load_prices(symbols)
print("rotation universe:", list(prices.columns), prices.shape)

rotation universe: ['QQQ', 'VGT', 'GLD', 'TLT', 'SPY', 'VIG'] (2122, 6)


## Generate weekly target weights

In [3]:
from strategies.multi_asset_rotation import MultiAssetRotation

weekly = prices.index[prices.index.weekday == 0]      # Mondays
strat = MultiAssetRotation()
W = strat.generate_weights(prices, weekly)
print("weekly target-weight panel:", W.shape)
assert bool((W.sum(axis=1) <= 1.0 + 1e-6).all())
W.tail(4).round(3)

Model is not converging.  Current: 4311.7548764670655 is not greater than 4337.492211584054. Delta is -25.737335116988106


Model is not converging.  Current: 4755.111891617247 is not greater than 4768.204137067607. Delta is -13.09224545035977


Model is not converging.  Current: 4836.127994911214 is not greater than 4837.498491663954. Delta is -1.3704967527401095


Model is not converging.  Current: 4851.142605545729 is not greater than 4870.572321344178. Delta is -19.42971579844925


Model is not converging.  Current: 4876.99485982178 is not greater than 4878.915166668805. Delta is -1.920306847025131


...

Model is not converging.  Current: 9600.378527971923 is not greater than 9608.235239116071. Delta is -7.856711144147994


Model is not converging.  Current: 9737.839819860179 is not greater than 9764.060091431329. Delta is -26.22027157115008


Model is not converging.  Current: 9784.569680435809 is not greater than 9787.049088305585. Delta is -2.4794078697759687


weekly target-weight panel: (397, 6)


,QQQ,VGT,GLD,TLT,SPY,VIG
date,,,,,,
2026-05-11,0.0,0.0,0.841,0.159,0.0,0.0
2026-05-18,0.0,0.0,0.372,0.128,0.0,0.0
2026-06-01,0.0,0.0,0.489,0.011,0.0,0.0
2026-06-08,0.0,0.0,0.000,0.000,0.0,0.0


## Run the backtest with transaction costs

In [4]:
from backtest.costs.transaction_costs import TransactionCostModel
from backtest.engines.vectorized import VectorizedBacktest

result = VectorizedBacktest(TransactionCostModel()).run(W, prices)
rets = result.returns.dropna()
print(result.summary())

{'total_return': -0.003397267951432248, 'cagr': -0.00040405079334804306, 'ann_vol': np.float64(0.08886880790826945), 'sharpe': 0.03997339661253118, 'max_drawdown': -0.18518398894785137}


## Tearsheet

In [5]:
from backtest.metrics.tearsheet import Tearsheet

ts = Tearsheet(rets)
metrics = ts.compute()
for k in ["cagr","ann_vol","sharpe","sortino","calmar","max_drawdown","var_95","cvar_95"]:
    print(f"{k:>14}: {metrics[k]:+.4f}")

          cagr: -0.0004
       ann_vol: +0.0889
        sharpe: +0.0400
       sortino: +0.0540
        calmar: -0.0022
  max_drawdown: -0.1852
        var_95: +0.0084
       cvar_95: +0.0150


## Deflated Sharpe Ratio (overfitting control)

In [6]:
from backtest.validation.deflated_sharpe import compute_dsr, probabilistic_sharpe_ratio

for n in [1, 10, 50, 200]:
    print(f"DSR (n_trials={n:>3}): {compute_dsr(rets, n_trials=n):.4f}")
print(f"PSR            : {probabilistic_sharpe_ratio(rets):.4f}")

DSR (n_trials=  1): 0.5461
DSR (n_trials= 10): 0.0723
DSR (n_trials= 50): 0.0154
DSR (n_trials=200): 0.0040
PSR            : 0.5461


## BHY multiple-testing correction across variants

In [7]:
from scipy import stats
from backtest.validation.multiple_testing import bhy_correction, MultipleTestingReport

variants = {"mom_63": 63, "mom_126": 126, "mom_252": 252}
pvals, labels = [], []
for label, lb in variants.items():
    s = MultiAssetRotation(mom_lookback=lb)
    w = s.generate_weights(prices, weekly)
    r = VectorizedBacktest(TransactionCostModel()).run(w, prices).returns.dropna()
    t = r.mean() / (r.std(ddof=1) + 1e-12) * np.sqrt(len(r))
    pvals.append(float(2 * (1 - stats.norm.cdf(abs(t))))); labels.append(label)
reject, adj = bhy_correction(pvals, alpha=0.05)
print(MultipleTestingReport(pvals, labels=labels).summary())

Model is not converging.  Current: 4336.992638680473 is not greater than 4337.006068270419. Delta is -0.01342958994609944


Model is not converging.  Current: 4755.111891617241 is not greater than 4768.451018125707. Delta is -13.339126508466506


Model is not converging.  Current: 4836.127994911218 is not greater than 4837.49851430408. Delta is -1.3705193928626613


Model is not converging.  Current: 4851.1426055457205 is not greater than 4870.572321344175. Delta is -19.42971579845471


Model is not converging.  Current: 4876.994859821773 is not greater than 4878.915166669063. Delta is -1.920306847289794


...

Model is not converging.  Current: 8871.524426305064 is not greater than 8874.442493929631. Delta is -2.918067624566902


Model is not converging.  Current: 9600.378527972229 is not greater than 9608.221063772315. Delta is -7.842535800085898


Model is not converging.  Current: 9737.839819860179 is not greater than 9764.060091431329. Delta is -26.22027157115008


Model is not converging.  Current: 9784.569680435809 is not greater than 9787.049088305585. Delta is -2.4794078697759687


Multiple-testing report (3 tests, alpha=0.05)
--------------------------------------------------------
  Raw significant (no correction): 0
  Benjamini-Hochberg (FDR):        0
  Benjamini-Hochberg-Yekutieli:    0
  Bonferroni (FWER):               0
--------------------------------------------------------
  Note: factor mining over many candidates inflates false
  positives; prefer BHY when strategy returns are correlated.


In [8]:
fig, ax = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
result.equity_curve.plot(ax=ax[0]); ax[0].set_title("Strategy equity"); ax[0].set_ylabel("NAV")
dd = ts.drawdown_series()
dd.plot(ax=ax[1], color="crimson"); ax[1].fill_between(dd.index, dd.values, 0, color="crimson", alpha=0.3)
ax[1].set_title("Drawdown"); plt.tight_layout(); print("backtest analysis complete.")

backtest analysis complete.
